# Create Qdrant Content Collection

## Init

### Imports

In [1]:
from qdrant_client import QdrantClient
from qdrant_client.models import VectorParams, Distance, PayloadSchemaType, PointStruct, SparseVectorParams, Document, Prefetch, FusionQuery
from qdrant_client import models

import pandas as pd
import os
from dotenv import load_dotenv
import openai

/Users/antoineestienne/GithubRepositories/ai-bootcamp-dev-repo/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Clients

In [2]:

load_dotenv()
qdrant_url = os.getenv("QDRANT_URL")
qdrant_api_key = os.getenv("QDRANT_API_KEY")
qdrant_client = QdrantClient(
    url=qdrant_url,
    api_key=qdrant_api_key
)

### Reusable functions

In [3]:
def get_embeddings_batch(text_list, model="text-embedding-3-small", batch_size=100):
    
    if len(text_list) <= batch_size:
        response = openai.embeddings.create(input=text_list, model=model)
        return [embedding.embedding for embedding in response.data]
    
    all_embeddings = []
    counter = 1
    for i in range(0, len(text_list), batch_size):
        batch = text_list[i:i + batch_size]
        response = openai.embeddings.create(input=batch, model=model)
        all_embeddings.extend([embedding.embedding for embedding in response.data])
        print(f"Processed {counter * batch_size} of {len(text_list)}")
        counter += 1
    
    return all_embeddings

## Create collection

In [4]:
CONTENT_COLLECTION_NAME="Content-collection-00"

In [5]:
# qdrant_client.create_collection(
#     collection_name=CONTENT_COLLECTION_NAME,
#     vectors_config={"text-embedding-3-small": VectorParams(size=1536, distance=Distance.COSINE)},
#     sparse_vectors_config={"bm25": SparseVectorParams(modifier=models.Modifier.IDF)}
# )

In [6]:
qdrant_client.create_payload_index(
    collection_name=CONTENT_COLLECTION_NAME,
    field_name="id",
    field_schema=PayloadSchemaType.KEYWORD
)

UpdateResult(operation_id=5, status=<UpdateStatus.COMPLETED: 'completed'>)

In [7]:
# define pydantic model for content payload
from pydantic import BaseModel
class Content(BaseModel):
    id: str
    content_text: str
    mediaType:str
    author:str

## Add medium articles

### Add first article, step by step

In [8]:
first_article = pd.read_json("content-files/01-the-guild-introduction.jsonl", lines=True)
first_article.head()


,article,section,text
0,The Guild — an introduction to a peer-run orga...,Overview of what The Guild is,The Guild is a peer-run organization where sof...
1,The Guild — an introduction to a peer-run orga...,Governance and structure,The organization should be flat and use member...
2,The Guild — an introduction to a peer-run orga...,Inspiration from historical guilds,Inspired by the artisan guilds of the past — w...
3,The Guild — an introduction to a peer-run orga...,Why the Guild matters,Decentralized technologies remind us that powe...
4,The Guild — an introduction to a peer-run orga...,The gap it fills,"Surprisingly, no such community exists yet at ..."


In [11]:
def preprocess_content_text(row):
    return f"Article title: {row['article']}\n Article section: {row['section']}\n Article content: {row['text']}"

In [12]:
# data_to_embed is a list of Content 
# preprocess_content_text for the content_text field
# id is a generated uuid
# mediaType is "article"
# author is "The Guild"
import uuid
data_to_embed = [Content(id=str(uuid.uuid4()), content_text=preprocess_content_text(row), mediaType="article", author="The Guild") for _, row in first_article.iterrows()]
data_to_embed[0]

Content(id='b85e3ccd-77cd-4136-8a45-6ec19db0bc96', content_text='Article title: The Guild — an introduction to a peer-run organization for developers\n Article section: Overview of what The Guild is\n Article content: The Guild is a peer-run organization where software developers certify each other’s skills, learn together, and create opportunities. It is built on the idea that developers are stronger when united.', mediaType='article', author='The Guild')

In [13]:
text_to_embed = [data.content_text for data in data_to_embed]

In [14]:
embeddings = get_embeddings_batch(text_to_embed)

In [20]:
pointstructs = []
i = 1
for embedding, data in zip(embeddings, data_to_embed):
    pointstructs.append(
        PointStruct(
            id=i,
            vector={
                "text-embedding-3-small": embedding,
                "bm25": Document(
                    text=data.content_text,
                    model="qdrant/bm25"
                )
            },
            # convert data to dict
            payload=data.model_dump()
        )
    )
    i += 1

In [21]:
pointstructs

[PointStruct(id=1, vector={'text-embedding-3-small': [-0.045501891523599625, 0.022264646366238594, -0.011618622578680515, 0.01303809229284525, 0.004626419860869646, -0.01567988283932209, -0.05709422752261162, 0.0658213347196579, 0.0012346429284662008, 0.001515579642727971, 0.04931342974305153, -0.037983957678079605, 0.007406214717775583, -0.02399955317378044, 0.05620048567652702, 0.03007172979414463, -0.04805167764425278, 0.00045015590148977935, -0.013563822023570538, 0.03493472561240196, 0.015548450872302055, 0.0021998495794832706, -0.015035864897072315, 0.07612563669681549, -0.001150033320300281, -0.022225216031074524, -0.023868121206760406, 0.01658676750957966, -0.014602137729525566, -0.005477444734424353, 0.004527845419943333, -0.015666740015149117, 0.02482757717370987, 0.004527845419943333, -0.004120405297726393, 0.042846955358982086, 0.025879036635160446, -0.004343840293586254, 0.02973000518977642, -0.01871597208082676, -0.02930942177772522, -0.03469814732670784, -0.0077742254361

In [23]:
qdrant_client.upsert(
    collection_name=CONTENT_COLLECTION_NAME,
    points=pointstructs
)

UpdateResult(operation_id=6, status=<UpdateStatus.COMPLETED: 'completed'>)

### Reusable function

In [ ]:
def process_and_upsert_article_content(df):
    data_to_embed = [Content(id=str(uuid.uuid4()), content_text=preprocess_content_text(row), mediaType="article", author="The Guild") for _, row in df.iterrows()]
    text_to_embed = [data.content_text for data in data_to_embed]
    embeddings = get_embeddings_batch(text_to_embed)
    pointstructs = []
    i = 1
    for embedding, data in zip(embeddings, data_to_embed):
        pointstructs.append(
            PointStruct(
                id=i,
                vector={
                    "text-embedding-3-small": embedding,
                    "bm25": Document(
                        text=data.content_text,
                        model="qdrant/bm25"
                    )
                },
                # convert data to dict
                payload=data.model_dump()
            )
        )
        i += 1
    qdrant_client.upsert(
        collection_name=CONTENT_COLLECTION_NAME,
        points=pointstructs
    )
    print(f"Upserted {len(pointstructs)} points for {df.shape[0]} rows")
    

### Add the other articles

In [ ]:
second_article = pd.read_json("content-files/02-the-guild-plan.jsonl", lines=True)
process_and_upsert_article_content(second_article)

Upserted 25 points for 25 rows


In [ ]:
third_article = pd.read_json("content-files/03-The Guild Genesis v0 is live build reputation together.jsonl", lines=True)
process_and_upsert_article_content(third_article)

Upserted 17 points for 17 rows


In [ ]:
fourth_article = pd.read_json("content-files/04-The Guild — October Update Building Together.jsonl", lines=True)
process_and_upsert_article_content(fourth_article)

Upserted 17 points for 17 rows
